In [1]:
import sys
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle
from numba import njit, prange
from matplotlib.animation import FuncAnimation,FFMpegWriter, PillowWriter
from IPython.display import HTML
from scipy.interpolate import RegularGridInterpolator
import time
import pandas as pd
from copy import deepcopy
import seaborn as sns
from numba import set_num_threads
from itertools import product
from tqdm import tqdm
from collections import defaultdict
import copy
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",              # Usa el serif por defecto (Computer Modern)
    "font.serif": ["Computer Modern Roman"],
    "font.size": 16,
    "axes.labelsize": 18,
    "axes.titlesize": 20,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14
})


set_num_threads(14)  # Reemplaza N con el número de núcleos que quieres usar

# Funcion general de simulacion

In [4]:
@njit(parallel=True, fastmath=True)
def actualizar_campos(presion_adversa, u_old, v_old, p_old, T_old, u_star, v_star, p_star, T_star,
                      Re, Pr, Ec, Eu, dt_star, dx_star, dy_star, nx, ny):
    for i in prange(1, ny - 1):
        for j in prange(1, nx - 1):
            conv_u_x = u_old[i, j] * (u_old[i, j + 1] - u_old[i, j - 1]) / (2 * dx_star)
            conv_u_y = v_old[i, j] * (u_old[i + 1, j] - u_old[i - 1, j]) / (2 * dy_star)
            diff_u_x = (u_old[i, j + 1] - 2 * u_old[i, j] + u_old[i, j - 1]) / dx_star**2
            diff_u_y = (u_old[i + 1, j] - 2 * u_old[i, j] + u_old[i - 1, j]) / dy_star**2
            u_star[i, j] += dt_star * (-conv_u_x - conv_u_y + (1 / Re) * (diff_u_x + diff_u_y))

            conv_v_x = u_old[i, j] * (v_old[i, j + 1] - v_old[i, j - 1]) / (2 * dx_star)
            conv_v_y = v_old[i, j] * (v_old[i + 1, j] - v_old[i - 1, j]) / (2 * dy_star)
            diff_v_x = (v_old[i, j + 1] - 2 * v_old[i, j] + v_old[i, j - 1]) / dx_star**2
            diff_v_y = (v_old[i + 1, j] - 2 * v_old[i, j] + v_old[i - 1, j]) / dy_star**2
            grad_p_y = (p_old[i + 1, j] - p_old[i - 1, j]) / (2 * dy_star)
            v_star[i, j] += dt_star * (-conv_v_x - conv_v_y - Eu * grad_p_y + (1 / Re) * (diff_v_x + diff_v_y))

    for _ in range(20):
        for i in prange(1, ny - 1):
            for j in prange(1, nx - 1):
                div = (u_star[i, j + 1] - u_star[i, j - 1]) / (2 * dx_star) + \
                      (v_star[i + 1, j] - v_star[i - 1, j]) / (2 * dy_star)
                p_star[i, j] = 0.25 * (p_old[i + 1, j] + p_old[i - 1, j] +
                                       p_old[i, j + 1] + p_old[i, j - 1] -
                                       (dx_star * dy_star) / (2 * (dx_star**2 + dy_star**2)) * div)
    
    # Gradiente de presión controlado
    for j in prange(nx):
        x = j * dx_star
        for i in prange(ny):
            if x <= 0.3:
                p_star[i, j] += 0.01  # presión constante en entrada
            else:
                p_star[i, j] += presion_adversa * (x - 0.3)


    for i in prange(1, ny - 1):
        for j in prange(1, nx - 1):
            u_star[i, j] -= dt_star * Eu * (p_star[i, j + 1] - p_star[i, j - 1]) / (2 * dx_star)
            v_star[i, j] -= dt_star * Eu * (p_star[i + 1, j] - p_star[i - 1, j]) / (2 * dy_star)

    for i in prange(1, ny - 1):
        for j in prange(1, nx - 1):
            conv_T_x = u_star[i, j] * (T_old[i, j + 1] - T_old[i, j - 1]) / (2 * dx_star)
            conv_T_y = v_star[i, j] * (T_old[i + 1, j] - T_old[i - 1, j]) / (2 * dy_star)
            diff_T_x = (T_old[i, j + 1] - 2 * T_old[i, j] + T_old[i, j - 1]) / dx_star**2
            diff_T_y = (T_old[i + 1, j] - 2 * T_old[i, j] + T_old[i - 1, j]) / dy_star**2
            Sxx = (u_star[i, j + 1] - u_star[i, j - 1]) / (2 * dx_star)
            Syy = (v_star[i + 1, j] - v_star[i - 1, j]) / (2 * dy_star)
            Sxy = 0.5 * ((u_star[i + 1, j] - u_star[i - 1, j]) / (2 * dy_star) +
                         (v_star[i, j + 1] - v_star[i, j - 1]) / (2 * dx_star))
            viscous_heating = (Ec / Re) * (2 * (Sxx**2 + Syy**2) + 4 * Sxy**2)
            T_star[i, j] += dt_star * (-conv_T_x - conv_T_y +
                                       (1 / (Re * Pr)) * (diff_T_x + diff_T_y) +
                                       viscous_heating)

    tau = np.zeros((ny, nx))
    for i in prange(1, ny - 1):
        for j in prange(nx):
            tau[i, j] = (u_star[i + 1, j] - u_star[i - 1, j]) / (2 * dy_star)

# Condiciones de frontera
    v_star[0, :] = 0
    v_star[-1, :] = 0
    u_star[0, :] = -0.5
    u_star[-1, :] = 1.0
    u_star[:, -1] = u_star[:, -2]
    p_star[:, 0] = p_star[:, 1]
    p_star[:, -1] = p_star[:, -2]
    p_star[0, :] = p_star[1, :]
    p_star[-1, :] = p_star[-2, :]

    return u_star, v_star, p_star, T_star, tau

def run_simulacion_general(nx=25, ny=25, presion_adversa=0.3,
                           observaciones_dict=None,
                           metodo_asimilacion=None,
                           variables_asimilar=['T'],
                           metodo_kwargs={},
                           N_FRAMES=300):
    """
    Simulación general que permite asimilación de datos opcional (sin guardado en disco).

    Parámetros:
    -----------
    nx, ny : int
        Resolución espacial.
    presion_adversa : float
        Presión aplicada en la entrada.
    observaciones_dict : dict
        Diccionario {timestep: observaciones}.
    metodo_asimilacion : function or None
        Función que aplica asimilación (e.g., aplicar_nudging).
    variables_asimilar : list of str
        Variables a asimilar (e.g., ['T', 'p']).
    metodo_kwargs : dict
        Parámetros adicionales para el método de asimilación.
    N_FRAMES : int
        Número de frames a guardar (espaciados uniformemente).

    Retorna:
    --------
    dict con históricos y parámetros.
    """

    # --- Parámetros físicos y temporales ---
    Re, Pr, Ec, Eu = 20.0, 10.0, 0.1, 1.0
    Lx_star = Ly_star = 1.0
    dx_star = Lx_star / (nx - 1)
    dy_star = Ly_star / (ny - 1)
    dt_cfl = 0.008 * dx_star / 1.0
    nt = int(np.ceil(2.0 / dt_cfl))
    dt_star = 1.0 / nt

    save_interval = max(1, nt // N_FRAMES)
    save_times = set(range(0, nt, save_interval))
    save_times.add(nt - 1)

    x_star = np.linspace(0, Lx_star, nx)
    y_star = np.linspace(0, Ly_star, ny)
    X_star, Y_star = np.meshgrid(x_star, y_star)

    # --- Condiciones iniciales ---
    u0, up = 1.0, -0.5
    T0_star, T1_star = 0.0, 0.0
    u_star = np.ones((ny, nx)) * u0
    v_star = np.zeros((ny, nx))
    p_star = np.zeros((ny, nx))
    T_star = np.ones((ny, nx)) * T1_star
    u_star[0, :] = up
    u_star[-1, :] = u0
    T_star[0, :] = T0_star
    T_star[-1, :] = T1_star

    u_hist, v_hist, p_hist, T_hist, tau_hist = [], [], [], [], []
    total_assim_time = 0.0

    for n in range(nt):
        u_old, v_old, p_old, T_old = u_star.copy(), v_star.copy(), p_star.copy(), T_star.copy()
        u_star, v_star, p_star, T_star, tau = actualizar_campos(
            presion_adversa, u_old, v_old, p_old, T_old,
            u_star, v_star, p_star, T_star,
            Re, Pr, Ec, Eu, dt_star, dx_star, dy_star, nx, ny
        )

        # Asimilación si corresponde
        if observaciones_dict and n in observaciones_dict and metodo_asimilacion:
            t0 = time.time()
            obs_t = observaciones_dict[n]
            for var in variables_asimilar:
                if var in obs_t:
                    campo = {'T': T_star, 'p': p_star, 'tau': tau}[var]
                    campo_asim = metodo_asimilacion(
                        campo_modelo=campo,
                        observaciones=obs_t,
                        variable=var,
                        nx=nx, ny=ny,
                        **metodo_kwargs
                    )
                    if var == 'T': T_star = campo_asim
                    elif var == 'p': p_star = campo_asim
                    elif var == 'tau': tau = campo_asim
            total_assim_time += time.time() - t0

        # Guardar en memoria
        if n in save_times:
            u_hist.append(u_star.copy())
            v_hist.append(v_star.copy())
            p_hist.append(p_star.copy())
            T_hist.append(T_star.copy())
            tau_hist.append(tau.copy())

    print("\n✅ Simulación finalizada.")

    return {
        "u_history": u_hist,
        "v_history": v_hist,
        "p_history": p_hist,
        "T_history": T_hist,
        "tau_history": tau_hist,
        "params": {
            "nx": nx, "ny": ny, "dt_star": dt_star, "nt": nt,
            "presion_adversa": presion_adversa,
            "metodo_asimilacion": metodo_asimilacion.__name__ if metodo_asimilacion else None,
            "variables_asimilar": variables_asimilar
        },
        "X_star": X_star,
        "Y_star": Y_star,
        "asimilacion_time": total_assim_time
    }



# Nudging

In [5]:
def aplicar_nudging(campo_modelo, observaciones, variable, nx, ny, alpha=0.1):
    """
    Aplica el método de Nudging para corregir un campo usando observaciones puntuales en y=0.

    Parámetros:
    -----------
    campo_modelo : ndarray
        Campo actual del modelo (T, p o tau), de tamaño (ny, nx).

    observaciones : dict
        Diccionario con las observaciones en este timestep.
        Debe contener claves como 'T', 'p', 'tau' y 'x_phys'.

    variable : str
        Nombre de la variable a asimilar: 'T', 'p' o 'tau'.

    nx, ny : int
        Tamaño espacial del dominio.

    alpha : float
        Coeficiente de nudging (cuánto se ajusta hacia la observación).

    Retorna:
    --------
    ndarray con el campo corregido.
    """
    campo_corr = campo_modelo.copy()

    if variable not in observaciones or 'x_phys' not in observaciones:
        return campo_corr

    x_phys = np.array(observaciones['x_phys'])
    x_idx = (x_phys * (nx - 1)).astype(int)
    y_idx = np.zeros_like(x_idx)  # Asumimos observaciones en y=0

    for xi, yi, obs_val in zip(x_idx, y_idx, observaciones[variable]):
        campo_corr[yi, xi] = (1 - alpha) * campo_corr[yi, xi] + alpha * obs_val

    return campo_corr


# Kalman filter

In [6]:
def aplicar_kalman_filter(campo_modelo, observaciones, variable, nx, ny,
                          sigma_obs=0.01, sigma_modelo=1e-4):
    """
    Aplica el filtro de Kalman clásico (KF) para una variable en y=0.

    Parámetros:
    -----------
    campo_modelo : np.ndarray
        Campo actual del modelo (e.g., T_star, p_star, tau).

    observaciones : dict
        Diccionario con observaciones. Debe incluir:
        - 'x_phys': posiciones físicas de los sensores
        - variable: valores observados para esa variable

    variable : str
        Nombre de la variable a asimilar ('T', 'p' o 'tau').

    nx, ny : int
        Dimensiones del dominio.

    sigma_obs : float
        Desviación estándar del ruido de observación.

    sigma_modelo : float
        Incertidumbre del modelo (ruido del sistema).

    Retorna:
    --------
    campo_actualizado : np.ndarray
        Campo corregido en base a las observaciones.
    """
    campo_actualizado = campo_modelo.copy()
    x_phys = observaciones["x_phys"]
    obs_vals = observaciones[variable]

    x_idx = (x_phys * (nx - 1)).astype(int)
    y_idx = np.zeros_like(x_idx)  # sensores en y = 0

    # Extraer predicciones del modelo
    x_fondo = campo_modelo[y_idx, x_idx]

    # Construir matrices
    H = np.eye(len(x_idx))  # observación directa
    R = np.eye(len(x_idx)) * sigma_obs**2
    P = np.eye(len(x_idx)) * sigma_modelo**2

    # Ganancia de Kalman
    K = P @ np.linalg.inv(P + R)

    # Corrección
    x_actualizado = x_fondo + K @ (obs_vals - H @ x_fondo)

    # Insertar corrección en el campo
    for xi, yi, val in zip(x_idx, y_idx, x_actualizado):
        campo_actualizado[yi, xi] = val

    return campo_actualizado


# Ensemble Kalman filter

Necesita su propia simulacion porque se añaden varios 'hilos' de simulacion. No solo se usa un valor inicial y se simula su evolucion (teniendo en cuenta la asimilacion), sino que genera N valores iniciales y simula su evolucion simultaneamente, formando el ensemble.

In [7]:
def run_enkf(nx=25, ny=25, N_ens=10, presion_adversa=0.3,
             observaciones_dict=None,
             variables_asimilar=['T'],
             sigma_obs=0.01, sigma_modelo=1e-4,
             N_FRAMES=300, seed=None):
    """
    Simulación con Ensemble Kalman Filter (EnKF).

    Parámetros:
    -----------
    N_ens : int
        Número de miembros del ensamble.
    sigma_obs : float
        Desviación estándar del ruido de observación.
    """
    if seed is not None:
        np.random.seed(seed)

    # Parámetros del modelo
    Re, Pr, Ec, Eu = 20.0, 10.0, 0.1, 1.0
    Lx_star = Ly_star = 1.0
    dx_star = Lx_star / (nx - 1)
    dy_star = Ly_star / (ny - 1)
    dt_cfl = 0.008 * dx_star / 1.0
    nt = int(np.ceil(2.0 / dt_cfl))
    dt_star = 1.0 / nt

    save_interval = max(1, nt // N_FRAMES)
    save_times = set(range(0, nt, save_interval))
    save_times.add(nt - 1)

    x_star = np.linspace(0, Lx_star, nx)
    y_star = np.linspace(0, Ly_star, ny)
    X_star, Y_star = np.meshgrid(x_star, y_star)

    # Estado inicial
    def crear_estado():
        u = np.ones((ny, nx)) * 1.0
        v = np.zeros((ny, nx))
        p = np.zeros((ny, nx))
        T = np.zeros((ny, nx))
        tau = np.zeros((ny, nx))
        u[0, :] = -0.5
        u[-1, :] = 1.0
        return {'u': u, 'v': v, 'p': p, 'T': T, 'tau': tau}

    ensamble = [crear_estado() for _ in range(N_ens)]

    u_hist, v_hist, p_hist, T_hist, tau_hist = [], [], [], [], []
    total_assim_time = 0.0

    for n in range(nt):
        for e in range(N_ens):
            m = ensamble[e]
            u, v, p, T, tau = m['u'], m['v'], m['p'], m['T'], m['tau']
            u_new, v_new, p_new, T_new, tau_new = actualizar_campos(
                presion_adversa, u, v, p, T,
                u.copy(), v.copy(), p.copy(), T.copy(),
                Re, Pr, Ec, Eu, dt_star, dx_star, dy_star, nx, ny
            )
            ensamble[e] = {'u': u_new, 'v': v_new, 'p': p_new, 'T': T_new, 'tau': tau_new}

        if observaciones_dict and n in observaciones_dict:
            t0 = time.time()
            obs_t = observaciones_dict[n]

            for var in variables_asimilar:
                x_phys = obs_t["x_phys"]
                x_idx = (x_phys * (nx - 1)).astype(int)
                y_idx = np.zeros_like(x_idx)
                Y_obs = obs_t[var]

                # Actualización punto por punto (más estable)
                for i, (xi, yi) in enumerate(zip(x_idx, y_idx)):
                    valores_ens = np.array([m[var][yi, xi] for m in ensamble])
                    var_ens = np.var(valores_ens)
                    if var_ens == 0:
                        continue  # evitar división por cero

                    # Ganancia de Kalman escalar
                    K = var_ens / (var_ens + sigma_obs**2)

                    for e in range(N_ens):
                        perturb = np.random.normal(0, sigma_obs)
                        innov = (Y_obs[i] + perturb) - ensamble[e][var][yi, xi]
                        ensamble[e][var][yi, xi] += K * innov

            total_assim_time += time.time() - t0

        if n in save_times:
            u_hist.append(np.mean([m['u'] for m in ensamble], axis=0))
            v_hist.append(np.mean([m['v'] for m in ensamble], axis=0))
            p_hist.append(np.mean([m['p'] for m in ensamble], axis=0))
            T_hist.append(np.mean([m['T'] for m in ensamble], axis=0))
            tau_hist.append(np.mean([m['tau'] for m in ensamble], axis=0))

    print("\n✅ Simulación EnKF finalizada.")

    return {
        "u_history": u_hist,
        "v_history": v_hist,
        "p_history": p_hist,
        "T_history": T_hist,
        "tau_history": tau_hist,
        "params": {
            "nx": nx, "ny": ny, "dt_star": dt_star, "nt": nt,
            "presion_adversa": presion_adversa,
            "N_ens": N_ens,
            "sigma_obs": sigma_obs,
            "variables_asimilar": variables_asimilar
        },
        "X_star": X_star,
        "Y_star": Y_star,
        "asimilacion_time": total_assim_time
    }


# 4D-Var

También necesita su propia funcion de simulacion. El proceso es totalmente diferente, al tener que buscar el estado que hace minima la funcion J

In [8]:
def run_4dvar(nx=25, ny=25, presion_adversa=0.3,
                         observaciones_dict=None,
                         variables_asimilar=['T'],
                         N_FRAMES=300,
                         ventana_asimilacion=20,
                         max_iter=10,
                         alpha=0.05):
    """
    Simulación con asimilación de datos 4DVar (simplificada).

    Parámetros:
    -----------
    ventana_asimilacion : int
        Número de pasos dentro de cada ventana de optimización.
    max_iter : int
        Número de iteraciones de ajuste por ventana.
    alpha : float
        Paso de gradiente descendente.
    """

    # --- Parámetros físicos y temporales ---
    Re, Pr, Ec, Eu = 20.0, 10.0, 0.1, 1.0
    Lx_star = Ly_star = 1.0
    dx_star = Lx_star / (nx - 1)
    dy_star = Ly_star / (ny - 1)
    dt_cfl = 0.008 * dx_star / 1.0
    nt = int(np.ceil(2.0 / dt_cfl))
    dt_star = 1.0 / nt

    save_interval = max(1, nt // N_FRAMES)
    save_times = set(range(0, nt, save_interval))
    save_times.add(nt - 1)

    x_star = np.linspace(0, Lx_star, nx)
    y_star = np.linspace(0, Ly_star, ny)
    X_star, Y_star = np.meshgrid(x_star, y_star)

    # --- Condiciones iniciales ---
    def crear_estado_inicial():
        u = np.ones((ny, nx)) * 1.0
        v = np.zeros((ny, nx))
        p = np.zeros((ny, nx))
        T = np.zeros((ny, nx))
        tau = np.zeros((ny, nx))
        u[0, :] = -0.5
        u[-1, :] = 1.0
        return u, v, p, T, tau

    u_star, v_star, p_star, T_star, tau = crear_estado_inicial()

    u_hist, v_hist, p_hist, T_hist, tau_hist = [], [], [], [], []
    total_assim_time = 0.0

    n = 0
    while n < nt:
        t0 = time.time()

        # 1. Guardar estado inicial de la ventana
        u0, v0, p0, T0, _ = u_star.copy(), v_star.copy(), p_star.copy(), T_star.copy(), tau.copy()
        estados = []

        # 2. Simular hacia adelante
        for i in range(ventana_asimilacion):
            u_old, v_old, p_old, T_old = u_star.copy(), v_star.copy(), p_star.copy(), T_star.copy()
            u_star, v_star, p_star, T_star, tau = actualizar_campos(
                presion_adversa, u_old, v_old, p_old, T_old,
                u_star, v_star, p_star, T_star,
                Re, Pr, Ec, Eu, dt_star, dx_star, dy_star, nx, ny
            )
            estados.append((u_star.copy(), v_star.copy(), p_star.copy(), T_star.copy(), tau.copy()))

        # 3. Calcular gradiente e intentar mejorar T0 (simplificación: solo T)
        if observaciones_dict and metodo_observaciones_en_ventana(observaciones_dict, n, ventana_asimilacion):
            for _ in range(max_iter):
                T_adj = T0.copy()
                grad = np.zeros_like(T0)

                u_tmp, v_tmp, p_tmp, T_tmp = u0.copy(), v0.copy(), p0.copy(), T_adj.copy()
                for i in range(ventana_asimilacion):
                    u_tmp, v_tmp, p_tmp, T_tmp, _ = actualizar_campos(
                        presion_adversa, u_tmp.copy(), v_tmp.copy(), p_tmp.copy(), T_tmp.copy(),
                        u_tmp, v_tmp, p_tmp, T_tmp,
                        Re, Pr, Ec, Eu, dt_star, dx_star, dy_star, nx, ny
                    )
                    timestep = n + i
                    if timestep in observaciones_dict:
                        obs = observaciones_dict[timestep]
                        if 'T' in obs:
                            for x_phys, val in zip(obs['x_phys'], obs['T']):
                                j = int(x_phys * (nx - 1))
                                grad[0, j] += 2 * (T_tmp[0, j] - val)  # en y=0

                # Descenso de gradiente (solo en y=0)
                T0[0, :] -= alpha * grad[0, :]

            # Re-simular con T corregida
            u_star, v_star, p_star, T_star, tau = u0.copy(), v0.copy(), p0.copy(), T0.copy(), tau.copy()
            estados = []
            for i in range(ventana_asimilacion):
                u_old, v_old, p_old, T_old = u_star.copy(), v_star.copy(), p_star.copy(), T_star.copy()
                u_star, v_star, p_star, T_star, tau = actualizar_campos(
                    presion_adversa, u_old, v_old, p_old, T_old,
                    u_star, v_star, p_star, T_star,
                    Re, Pr, Ec, Eu, dt_star, dx_star, dy_star, nx, ny
                )
                estados.append((u_star.copy(), v_star.copy(), p_star.copy(), T_star.copy(), tau.copy()))

        total_assim_time += time.time() - t0

        # 4. Guardar
        for i in range(ventana_asimilacion):
            if (n + i) in save_times:
                u_hist.append(estados[i][0])
                v_hist.append(estados[i][1])
                p_hist.append(estados[i][2])
                T_hist.append(estados[i][3])
                tau_hist.append(estados[i][4])

        n += ventana_asimilacion

    print("\n✅ Simulación 4DVar finalizada.")

    return {
        "u_history": u_hist,
        "v_history": v_hist,
        "p_history": p_hist,
        "T_history": T_hist,
        "tau_history": tau_hist,
        "params": {
            "nx": nx, "ny": ny, "dt_star": dt_star, "nt": nt,
            "presion_adversa": presion_adversa,
            "ventana_asimilacion": ventana_asimilacion,
            "max_iter": max_iter
        },
        "X_star": X_star,
        "Y_star": Y_star,
        "asimilacion_time": total_assim_time
    }
def metodo_observaciones_en_ventana(observaciones_dict, n_ini, ventana):
    """Devuelve True si hay al menos una observación en la ventana."""
    return any((n_ini + i) in observaciones_dict for i in range(ventana))


# Observaciones

In [9]:
def extraer_observaciones(
    path_pkl,
    N_sens=10,
    T_sens=5,
    ruido_std=0.0,
    seed=None
):
    """
    Extrae observaciones desde un archivo .pkl de simulación (normalizado o no),
    tomando datos en la línea y = 0.1.

    Parámetros:
    - path_pkl: ruta al archivo .pkl con los datos simulados/interpolados
    - N_sens: número de sensores a lo largo de x
    - T_sens: número de pasos temporales donde tomar observaciones
    - ruido_std: desviación estándar del ruido (gaussiano)
    - seed: semilla para reproducibilidad

    Retorna:
    - Lista de diccionarios con observaciones por instante
    """

    if seed is not None:
        np.random.seed(seed)

    with open(path_pkl, "rb") as f:
        datos = pickle.load(f)

    T_hist = datos["T_history"]
    p_hist = datos["p_history"]
    tau_hist = datos["tau_history"]
    Y_star = datos["Y_star"]

    nt = len(T_hist)
    nx = T_hist[0].shape[1]
    x_physical = np.linspace(0, 1, N_sens)
    x_idx = (x_physical * (nx - 1)).astype(int)
    frame_indices = np.linspace(0, nt - 1, T_sens, dtype=int)

    # Elegir fila más cercana a y = 0.1
    y_fila = Y_star[:, 0]
    y_idx = int(np.argmin(np.abs(y_fila - 0.1)))

    observaciones = []
    for t in frame_indices:
        T_line = T_hist[t][y_idx, :]
        p_line = p_hist[t][y_idx, :]
        tau_line = tau_hist[t][y_idx, :]

        obs_t = {
            "t": t,
            "x_phys": x_physical,
            "T": T_line[x_idx].copy(),
            "p": p_line[x_idx].copy(),
            "tau": tau_line[x_idx].copy()
        }

        if ruido_std > 0.0:
            obs_t["T"] += np.random.normal(0, ruido_std, size=N_sens)
            obs_t["p"] += np.random.normal(0, ruido_std, size=N_sens)
            obs_t["tau"] += np.random.normal(0, ruido_std, size=N_sens)

        observaciones.append(obs_t)

    return observaciones


# Generacion de datos

Se generan los 300 frames de la evolucion para una resolucion de 25x25 usando datos interpolados de la resolucion 800x800 como observaciones en la placa.

In [10]:
EJECUTAR_SIMULACIONES = True  # 🔁 Cambia a True si quieres que esta celda se ejecute

if EJECUTAR_SIMULACIONES:

    # --- Cargar referencia ---
    with open("sim_normal/25_bloques.pkl", "rb") as f:
        ref_data = pickle.load(f)

    # --- Parámetros comunes ---
    resolucion = (25, 25)
    res_str = f"{resolucion[0]}"
    carpeta_base = f"DA/{res_str}_y_10%"
    os.makedirs(carpeta_base, exist_ok=True)

    metodos_especiales = ["enkf", "4dvar"]
    metodos_asimilacion = {
        "nudging": aplicar_nudging,
        "kf": aplicar_kalman_filter
    }

    variables_asimilar_set = [
        ["T"],         # solo temperatura
        ["p"],         # solo presión
        ["tau"],       # solo shear stress
        ["T", "p"],
        ["T", "tau"],
        ["T", "p", "tau"]
    ]

    n_sensores_set = [5, 10, 20]
    frecuencias_set = [300, 150, 70]

    # --- Bucle de configuraciones ---
    tag_map = {"T": "t", "p": "p", "tau": "s"}  # Etiquetas únicas y claras

    for variables_asimilar, n_sens, freq in product(variables_asimilar_set, n_sensores_set, frecuencias_set):

        print(f"\n🔧 Configuración: Vars={variables_asimilar}, Sensores={n_sens}, Freq={freq}")

        observaciones = extraer_observaciones(
        path_pkl="sim_normal/25_bloques.pkl",
        N_sens=n_sens,
        T_sens=freq,
        ruido_std=0.01,
        seed=42
    )

        observaciones_dict = {obs["t"]: obs for obs in observaciones}

        # ✅ Etiqueta segura para archivo
        vars_tag = ''.join(sorted(set(tag_map[v] for v in variables_asimilar)))

        # Métodos generales (nudging, kf)
        for nombre_metodo, funcion_asimilacion in metodos_asimilacion.items():
            print(f"🔄 Ejecutando {nombre_metodo.upper()}...")
            resultado = run_simulacion_general(
                nx=resolucion[0],
                ny=resolucion[1],
                presion_adversa=0.3,
                observaciones_dict=observaciones_dict,
                metodo_asimilacion=funcion_asimilacion,
                variables_asimilar=variables_asimilar,
                metodo_kwargs={},
                N_FRAMES=300
            )

            nombre_archivo = f"{nombre_metodo}_{res_str}_obs_{n_sens}_freq_{freq}_var_{vars_tag}.pkl"
            with open(os.path.join(carpeta_base, nombre_archivo), "wb") as f:
                pickle.dump(resultado, f)
            print(f"✅ Guardado: {nombre_archivo}")

        # EnKF
        print("🔄 Ejecutando ENKF...")
        resultado_enkf = run_enkf(
            nx=resolucion[0],
            ny=resolucion[1],
            N_ens=10,
            presion_adversa=0.3,
            observaciones_dict=observaciones_dict,
            variables_asimilar=variables_asimilar,
            sigma_obs=0.01,
        )
        nombre_enkf = f"enkf_{res_str}_obs_{n_sens}_freq_{freq}_var_{vars_tag}.pkl"
        with open(os.path.join(carpeta_base, nombre_enkf), "wb") as f:
            pickle.dump(resultado_enkf, f)
        print(f"✅ Guardado: {nombre_enkf}")

        # 4DVar
        print("🔄 Ejecutando 4DVAR...")
        resultado_4dvar = run_4dvar(
            nx=resolucion[0],
            ny=resolucion[1],
            presion_adversa=0.3,
            observaciones_dict=observaciones_dict,
            variables_asimilar=variables_asimilar,
            N_FRAMES=300,
            ventana_asimilacion=20,
            max_iter=10,
            alpha=0.05
        )
        nombre_4dvar = f"4dvar_{res_str}_obs_{n_sens}_freq_{freq}_var_{vars_tag}.pkl"
        with open(os.path.join(carpeta_base, nombre_4dvar), "wb") as f:
            pickle.dump(resultado_4dvar, f)
        print(f"✅ Guardado: {nombre_4dvar}")

# 23/07/2025 ---> tardó 388 minutos en ejecutar todo el script con las simulaciones con asimilación



🔧 Configuración: Vars=['T'], Sensores=5, Freq=300
🔄 Ejecutando NUDGING...

✅ Simulación finalizada.
✅ Guardado: nudging_25_obs_5_freq_300_var_t.pkl
🔄 Ejecutando KF...

✅ Simulación finalizada.
✅ Guardado: kf_25_obs_5_freq_300_var_t.pkl
🔄 Ejecutando ENKF...

✅ Simulación EnKF finalizada.
✅ Guardado: enkf_25_obs_5_freq_300_var_t.pkl
🔄 Ejecutando 4DVAR...

✅ Simulación 4DVar finalizada.
✅ Guardado: 4dvar_25_obs_5_freq_300_var_t.pkl

🔧 Configuración: Vars=['T'], Sensores=5, Freq=150
🔄 Ejecutando NUDGING...

✅ Simulación finalizada.
✅ Guardado: nudging_25_obs_5_freq_150_var_t.pkl
🔄 Ejecutando KF...

✅ Simulación finalizada.
✅ Guardado: kf_25_obs_5_freq_150_var_t.pkl
🔄 Ejecutando ENKF...

✅ Simulación EnKF finalizada.
✅ Guardado: enkf_25_obs_5_freq_150_var_t.pkl
🔄 Ejecutando 4DVAR...

✅ Simulación 4DVar finalizada.
✅ Guardado: 4dvar_25_obs_5_freq_150_var_t.pkl

🔧 Configuración: Vars=['T'], Sensores=5, Freq=70
🔄 Ejecutando NUDGING...

✅ Simulación finalizada.
✅ Guardado: nudging_25_obs_5_fre

# 📊 Análisis de Configuraciones de Asimilación de Datos

Este notebook analiza todas las combinaciones generadas para:
- Métodos de asimilación: `nudging`, `kf`, `enkf`, `4dvar`
- Variables asimiladas: `T`, `p`, `tau`, y sus combinaciones
- Número de sensores: `5`, `10`, `20`
- Frecuencia de observación: `70`, `150`, `300` (→ mayor valor = más observaciones)

Se identifican por cada método:
- 🏆 La combinación **más precisa** (menor RMSE medio)
- ⚡ La combinación **más eficiente** (menor tiempo de asimilación)

Además se generan animaciones para cada una de ellas.

---

In [12]:
# --- Cargar referencia interpolada a 25x25 ---
with open("sim_normal/25_bloques.pkl", "rb") as f:
    ref_interp = pickle.load(f)

# --- Función de RMSE y normalización ---
def rmse(a, b):
    return np.sqrt(np.mean((a - b) ** 2))

def normalizar_global(frames):
    all_vals = np.concatenate([f.ravel() for f in frames])
    vmin, vmax = np.min(all_vals), np.max(all_vals)
    return [(f - vmin) / (vmax - vmin + 1e-12) for f in frames]

def calcular_rmse_variable(sim_data, ref_data, var_name):
    sim = sim_data.get(var_name)
    ref = ref_data.get(var_name)
    if sim is None or ref is None:
        return np.nan
    sim_norm = normalizar_global(sim)
    return np.mean([rmse(s, r) for s, r in zip(sim_norm, ref)])

# --- Parsear nombre con nueva convención ---
def parsear_nombre(nombre):
    partes = nombre.replace(".pkl", "").split("_")
    return {
        "archivo": nombre,
        "metodo": partes[0],
        "n_sensores": int(partes[3]),
        "frecuencia": int(partes[5]),
        "variables": "".join(sorted(partes[7]))  # ej: 'tp', 'ts', 'pst', etc.
    }

# --- Cargar resultados de DA ---
base_dir = "DA/25_y_10%"
archivos = [f for f in os.listdir(base_dir)
            if f.endswith(".pkl") and not f.startswith("sim_") and "referencia" not in f]

# --- Ejecutar análisis ---
resultados = []
for archivo in archivos:
    ruta = os.path.join(base_dir, archivo)
    with open(ruta, "rb") as f:
        sim = pickle.load(f)

    meta = parsear_nombre(archivo)
    meta["rmse_T"] = calcular_rmse_variable(sim, ref_interp, "T_history")
    meta["rmse_p"] = calcular_rmse_variable(sim, ref_interp, "p_history")
    meta["rmse_tau"] = calcular_rmse_variable(sim, ref_interp, "tau_history")
    
    valores = [meta[k] for k in ["rmse_T", "rmse_p", "rmse_tau"] if not np.isnan(meta[k])]
    meta["rmse_total"] = np.mean(valores) if valores else np.nan
    meta["tiempo"] = sim.get("asimilacion_time", np.nan)
    resultados.append(meta)

# --- Exportar resultados ---
df = pd.DataFrame(resultados)
df.to_csv("analisis_resultados_DA.csv", index=False)
print("✅ Guardado: analisis_resultados_DA.csv")

# --- Mostrar resumen de mejores por método ---
mas_precisos = df.sort_values("rmse_total").groupby("metodo").first().reset_index()
mas_eficientes = df.sort_values("tiempo").groupby("metodo").first().reset_index()

print("\n🏆 Más precisos por método:")
print(mas_precisos[["metodo", "variables", "n_sensores", "frecuencia", "rmse_total", "tiempo"]])

print("\n⚡ Más eficientes por método:")
print(mas_eficientes[["metodo", "variables", "n_sensores", "frecuencia", "rmse_total", "tiempo"]])


FileNotFoundError: [Errno 2] No such file or directory: 'DA/25_y_10%'

In [13]:
dx_star = 1 / (25 - 1)
dy_star = 1 / (25 - 1)
dt_cfl = 0.008 * dx_star / 1.0
nt = int(np.ceil(2.0 / dt_cfl))
print('300 '+ str(nt/300))
print('150 '+ str(nt/150))
print('70 '+ str(int(nt/70)))

300 20.0
150 40.0
70 85


In [14]:
# Cargar los resultados desde el CSV
df = pd.read_csv("analisis_resultados_DA.csv")

# Eliminar filas con errores faltantes
df = df.dropna(subset=["rmse_total", "tiempo"])

# Obtener la configuración más precisa (menor RMSE total)
mas_preciso = df.loc[df["rmse_total"].idxmin()]

# Obtener la configuración más eficiente (menor tiempo)
mas_eficiente = df.loc[df["tiempo"].idxmin()]

# Mostrar resultados
print("🏆 Configuración más precisa:")
print(mas_preciso)

print("\n⚡ Configuración más eficiente:")
print(mas_eficiente)


🏆 Configuración más precisa:
archivo       4dvar_25_obs_5_freq_150_var_pst.pkl
metodo                                      4dvar
n_sensores                                      5
frecuencia                                    150
variables                                     pst
rmse_T                                   0.069933
rmse_p                                   0.040379
rmse_tau                                 0.263728
rmse_total                                0.12468
tiempo                                  71.284024
Name: 75, dtype: object

⚡ Configuración más eficiente:
archivo       nudging_25_obs_10_freq_70_var_s.pkl
metodo                                    nudging
n_sensores                                     10
frecuencia                                     70
variables                                       s
rmse_T                                   0.088038
rmse_p                                   0.040379
rmse_tau                                 0.263728
rmse_total     

In [15]:
import os
import pickle
import numpy as np
import pandas as pd

# --- Cargar referencia interpolada ---
with open("sim_normal/25_bloques.pkl", "rb") as f:
    ref_interp = pickle.load(f)

# --- Funciones de utilidad ---
def rmse(a, b):
    return np.sqrt(np.mean((a - b) ** 2))

def normalizar_global(frames):
    all_vals = np.concatenate([f.ravel() for f in frames])
    vmin, vmax = np.min(all_vals), np.max(all_vals)
    return [(f - vmin) / (vmax - vmin + 1e-12) for f in frames]

def calcular_rmse_velocidad(sim_data, ref_data):
    u_sim = np.array(sim_data["u_history"])
    v_sim = np.array(sim_data["v_history"])
    u_ref = np.array(ref_data["u_history"])
    v_ref = np.array(ref_data["v_history"])

    # Calcular |u| para cada frame
    vel_sim = np.sqrt(u_sim**2 + v_sim**2)
    vel_ref = np.sqrt(u_ref**2 + v_ref**2)

    # Normalizar magnitudes
    vel_sim_norm = normalizar_global(vel_sim)
    vel_ref_norm = normalizar_global(vel_ref)

    return np.mean([rmse(s, r) for s, r in zip(vel_sim_norm, vel_ref_norm)])

def parsear_nombre(nombre):
    partes = nombre.replace(".pkl", "").split("_")
    return {
        "archivo": nombre,
        "metodo": partes[0],
        "n_sensores": int(partes[3]),
        "frecuencia": int(partes[5]),
        "variables": "".join(sorted(partes[7]))  # ej: 'tp', 'ts', 'pst', etc.
    }

# --- Directorio de resultados ---
base_dir = "DA/25_y_10%"
archivos = [f for f in os.listdir(base_dir)
            if f.endswith(".pkl") and not f.startswith("sim_") and "referencia" not in f]

# --- Análisis ---
resultados = []
for archivo in archivos:
    ruta = os.path.join(base_dir, archivo)
    with open(ruta, "rb") as f:
        sim = pickle.load(f)

    meta = parsear_nombre(archivo)
    meta["rmse_velocidad"] = calcular_rmse_velocidad(sim, ref_interp)
    meta["tiempo"] = sim.get("asimilacion_time", np.nan)
    resultados.append(meta)

# --- Exportar ---
df = pd.DataFrame(resultados)
df.to_csv("analisis_resultados_velocidad.csv", index=False)
print("✅ Guardado: analisis_resultados_velocidad.csv")

# --- Mostrar resumen de mejores por método ---
mas_precisos = df.sort_values("rmse_velocidad").groupby("metodo").first().reset_index()
mas_eficientes = df.sort_values("tiempo").groupby("metodo").first().reset_index()

print("\n🏁 Mejores en velocidad por método:")
print(mas_precisos[["metodo", "variables", "n_sensores", "frecuencia", "rmse_velocidad", "tiempo"]])

print("\n⚡ Más eficientes por método:")
print(mas_eficientes[["metodo", "variables", "n_sensores", "frecuencia", "rmse_velocidad", "tiempo"]])


✅ Guardado: analisis_resultados_velocidad.csv

🏁 Mejores en velocidad por método:
    metodo variables  n_sensores  frecuencia  rmse_velocidad     tiempo
0    4dvar         p          20         150        0.236462  29.536269
1     enkf         s           5          70        0.236462   0.006768
2       kf         p          20         300        0.236460   0.139599
3  nudging        pt          20         300        0.233741   0.006728

⚡ Más eficientes por método:
    metodo variables  n_sensores  frecuencia  rmse_velocidad     tiempo
0    4dvar        pt           5         300        0.236462  26.885763
1     enkf         s          10          70        0.236462   0.006390
2       kf         s           5          70        0.236462   0.002265
3  nudging         s          10          70        0.236462   0.000847


In [18]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# === Función para crear animación ===
def animar_velocidad_tripleta(sim_bloques, sim_sin_da, sim_da, nombre_da, salida_path):
    u_bloques = np.array(sim_bloques["u_history"])
    v_bloques = np.array(sim_bloques["v_history"])
    u_sin_da = np.array(sim_sin_da["u_history"])
    v_sin_da = np.array(sim_sin_da["v_history"])
    u_da = np.array(sim_da["u_history"])
    v_da = np.array(sim_da["v_history"])

    vel_bloques = np.sqrt(u_bloques**2 + v_bloques**2)
    vel_sin_da = np.sqrt(u_sin_da**2 + v_sin_da**2)
    vel_da = np.sqrt(u_da**2 + v_da**2)

    # Asegurar que todos tengan la misma longitud
    nt = min(len(vel_bloques), len(vel_sin_da), len(vel_da))
    vel_bloques = vel_bloques[:nt]
    vel_sin_da = vel_sin_da[:nt]
    vel_da = vel_da[:nt]

    # Meshgrid (usamos el de bloques)
    X = sim_bloques["X_star"]
    Y = sim_bloques["Y_star"]

    # Rangos globales
    vmin = min(np.min(vel_bloques), np.min(vel_sin_da), np.min(vel_da))
    vmax = max(np.max(vel_bloques), np.max(vel_sin_da), np.max(vel_da))

    fig, axs = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
    titulos = ["25 bloques", "25 sin DA", f"DA: {nombre_da}"]

    def update(frame):
        datos = [vel_bloques[frame], vel_sin_da[frame], vel_da[frame]]
        for ax, data, titulo in zip(axs, datos, titulos):
            ax.clear()
            cf = ax.contourf(X, Y, data, levels=30, cmap="jet", vmin=vmin, vmax=vmax)
            ax.set_title(f"{titulo}\nt* = {frame / (nt - 1):.2f}", fontsize=14)
            ax.set_xlabel("x*", fontsize=12)
            ax.set_ylabel("y*", fontsize=12)
            ax.tick_params(labelsize=10)

        return []

    anim = FuncAnimation(fig, update, frames=nt, interval=100, blit=False)
    anim.save(salida_path, fps=20)
    plt.close()
    print(f"✅ Animación guardada: {salida_path}")

# === Configuraciones de DA seleccionadas por RMSE de velocidad ===
mejores_configs = {
    "KF_s_5_70": "DA/25/kf_25_obs_5_freq_70_var_s.pkl",
    "EnKF_s_10_70": "DA/25/enkf_25_obs_10_freq_70_var_s.pkl",
    "4DVAR_pt_5_300": "DA/25/4dvar_25_obs_5_freq_300_var_pt.pkl",
    "Nudging_s_10_70": "DA/25/nudging_25_obs_10_freq_70_var_s.pkl"
}

# === Rutas comunes ===
ruta_bloques = "sim_normal/25_bloques.pkl"
ruta_sin_da = "sim_normal/25.pkl"

# === Crear carpeta de salida ===
os.makedirs("anim_velocidad", exist_ok=True)

# === Cargar datos comunes una sola vez ===
with open(ruta_bloques, "rb") as f:
    sim_bloques = pickle.load(f)
with open(ruta_sin_da, "rb") as f:
    sim_sin_da = pickle.load(f)

# === Generar animaciones ===
for nombre, ruta_da in mejores_configs.items():
    with open(ruta_da, "rb") as f:
        sim_da = pickle.load(f)

    salida_path = os.path.join("anim_velocidad", f"velocidad_{nombre}.mp4")
    animar_velocidad_tripleta(sim_bloques, sim_sin_da, sim_da, nombre_da=nombre, salida_path=salida_path)


✅ Animación guardada: anim_velocidad/velocidad_KF_s_5_70.mp4
✅ Animación guardada: anim_velocidad/velocidad_EnKF_s_10_70.mp4
✅ Animación guardada: anim_velocidad/velocidad_4DVAR_pt_5_300.mp4
✅ Animación guardada: anim_velocidad/velocidad_Nudging_s_10_70.mp4
